In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("init_load_flag", "")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
# DimDate is not sourced from silver — it is programmatically generated
# from a date range that covers all historical and future orders
# Adjust start_date and end_date if your orders data has a wider range

start_date = "2020-01-01"
end_date   = "2030-12-31"

In [0]:
# Generate one row per date using spark.range
# spark.range gives us a sequence of longs (0, 1, 2 ...)
# we convert each long to a date by adding it as days to start_date

start_ts = to_date(lit(start_date))
end_ts   = to_date(lit(end_date))

total_days = spark.sql(f"select datediff(date('{end_date}'), date('{start_date}')) + 1 as n").collect()[0]['n']
print("total days to generate:", total_days)

df = spark.range(0, total_days).select(
    date_add(lit(start_date).cast("date"), col("id").cast("int")).alias("full_date")
)

In [0]:
df.limit(5).display()

In [0]:
# Build all date attributes from full_date
# DimDateKey is YYYYMMDD integer — standard industry pattern
# This lets FactOrders store a lightweight int FK instead of a full date string

df = df\
    .withColumn("DimDateKey",    date_format(col("full_date"), "yyyyMMdd").cast("int"))\
    .withColumn("year",          year(col("full_date")))\
    .withColumn("quarter",       quarter(col("full_date")))\
    .withColumn("month",         month(col("full_date")))\
    .withColumn("month_name",    date_format(col("full_date"), "MMMM"))\
    .withColumn("month_short",   date_format(col("full_date"), "MMM"))\
    .withColumn("week_of_year",  weekofyear(col("full_date")))\
    .withColumn("day_of_month",  dayofmonth(col("full_date")))\
    .withColumn("day_of_week",   dayofweek(col("full_date")))\
    .withColumn("day_name",      date_format(col("full_date"), "EEEE"))\
    .withColumn("day_short",     date_format(col("full_date"), "EEE"))\
    .withColumn("is_weekend",    when(dayofweek(col("full_date")).isin(1, 7), lit(1)).otherwise(lit(0)))\
    .withColumn("is_weekday",    when(dayofweek(col("full_date")).isin(1, 7), lit(0)).otherwise(lit(1)))\
    .withColumn("year_month",    date_format(col("full_date"), "yyyy-MM"))\
    .withColumn("year_quarter",  concat_ws("-Q", year(col("full_date")), quarter(col("full_date"))))\
    .withColumn("create_date",   current_timestamp())

In [0]:
# Put DimDateKey as the first column — standard convention for dimension keys
df = df.select(
    "DimDateKey",
    "full_date",
    "year",
    "quarter",
    "month",
    "month_name",
    "month_short",
    "week_of_year",
    "day_of_month",
    "day_of_week",
    "day_name",
    "day_short",
    "is_weekend",
    "is_weekday",
    "year_month",
    "year_quarter",
    "create_date"
)

In [0]:
df.limit(10).display()

In [0]:
print("total rows:", df.count())

In [0]:
# DimDate is a static dimension — no SCD, no expire/insert pattern needed
# init_load_flag == 1 : first run, table does not exist yet -> write overwrite
# init_load_flag == 0 : subsequent run -> merge on DimDateKey
#                       only inserts new dates (e.g. when range is extended)
#                       never updates existing rows since date attributes never change

if spark.catalog.tableExists("databricks_cata.gold.DimDate"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@customersprojectete.dfs.core.windows.net/DimDate")

    dlt_obj.alias("tgt").merge(
        df.alias("src"),
        "tgt.DimDateKey = src.DimDateKey"
    ).whenNotMatchedInsertAll().execute()

else:
    df.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/DimDate")\
        .saveAsTable("databricks_cata.gold.DimDate")

In [0]:
%sql
select * from databricks_cata.gold.DimDate order by DimDateKey limit 10

In [0]:
%sql
select count(*) as total_rows from databricks_cata.gold.DimDate

In [0]:
%sql
describe databricks_cata.silver.orders_silver

In [0]:
%sql
DESCRIBE databricks_cata.gold.dimcustomersscd2;

In [0]:
%sql
DESCRIBE databricks_cata.gold.dimproductsscd2;

In [0]:
%sql
DESCRIBE databricks_cata.gold.dimregion;

In [0]:
%sql
DESCRIBE databricks_cata.gold.dimdate;